# Week 5 Day 4 — Validate Predictive Power

Test whether the z-score signal actually predicts future residual convergence.

**Logic:**  
If a bond is cheap (z > 0), its residual should *fall* over the next N days (convergence).  
If a bond is rich (z < 0), its residual should *rise*.

So `z_score` vs `forward_residual_change` should have a **negative slope**.

We test at two horizons: **5-day** and **20-day**.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
%matplotlib inline

## 1. Load signal

In [ ]:
sig = pd.read_parquet('../data/processed/richness_signal.parquet')
sig['date'] = pd.to_datetime(sig['date'])

print(f"Shape : {sig.shape}")
print(f"Dates : {sig['date'].min().date()} -> {sig['date'].max().date()}")
print(f"Maturities: {sorted(sig['maturity'].unique())}")
sig.head()

## 2. Compute forward residual changes

We pivot to a wide matrix (date × maturity), then shift by N rows to get residual N *trading days* ahead.  
`fwd_N = residual(t+N) - residual(t)` — if the signal predicts convergence, high-z rows should have negative fwd values.

In [ ]:
# Wide matrix: rows = dates (sorted), columns = maturities
pivot = (
    sig.pivot(index='date', columns='maturity', values='residual_bps')
    .sort_index()
)

# shift(-N) aligns each row with the residual N trading days later
fwd5  = pivot.shift(-5)  - pivot
fwd20 = pivot.shift(-20) - pivot

# Melt back to long and rename
fwd5_long  = fwd5.stack(future_stack=True).rename('fwd5_bps').reset_index()
fwd20_long = fwd20.stack(future_stack=True).rename('fwd20_bps').reset_index()
fwd5_long.columns  = ['date', 'maturity', 'fwd5_bps']
fwd20_long.columns = ['date', 'maturity', 'fwd20_bps']

# Merge onto signal DataFrame
df = (
    sig
    .merge(fwd5_long,  on=['date', 'maturity'], how='left')
    .merge(fwd20_long, on=['date', 'maturity'], how='left')
    .dropna(subset=['z_score', 'fwd5_bps', 'fwd20_bps'])
)

print(f"Rows with valid z + both forward changes: {len(df):,}")
df[['date', 'maturity', 'residual_bps', 'z_score', 'fwd5_bps', 'fwd20_bps']].head(8)

## 3. Bin by z-score, compute mean forward change

We split observations into 10 equal-population bins by z-score.  
Each bin's mean forward change is our estimate of "expected convergence at that signal strength."

In [ ]:
df['z_bin'] = pd.qcut(df['z_score'], q=10, labels=False, duplicates='drop')

binned = (
    df.groupby('z_bin')
    .agg(
        z_mean     = ('z_score',   'mean'),
        z_low      = ('z_score',   'min'),
        z_high     = ('z_score',   'max'),
        fwd5_mean  = ('fwd5_bps',  'mean'),
        fwd20_mean = ('fwd20_bps', 'mean'),
        count      = ('z_score',   'count'),
    )
    .reset_index()
)

print(binned[['z_bin', 'z_mean', 'fwd5_mean', 'fwd20_mean', 'count']].round(3).to_string(index=False))

## 4. Main plot: signal strength vs forward convergence

**Pass criterion:** bars slope monotonically downward left-to-right.  
- Left bars (rich, z < 0) → positive change (residual rises → convergence for short)  
- Right bars (cheap, z > 0) → negative change (residual falls → convergence for long)  

**Fail criterion:** flat or random bars → the signal is noise, stop here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, col, label in [
    (axes[0], 'fwd5_mean',  '5-day forward'),
    (axes[1], 'fwd20_mean', '20-day forward'),
]:
    colors = ['firebrick' if v > 0 else 'steelblue' for v in binned[col]]
    ax.bar(range(len(binned)), binned[col], color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.9)

    # label x-axis with mean z in each bin
    ax.set_xticks(range(len(binned)))
    ax.set_xticklabels([f'{z:.2f}' for z in binned['z_mean']], rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Mean z-score in bin (rich ← → cheap)')
    ax.set_ylabel('Mean forward residual change (bp)')
    ax.set_title(f'{label} residual change by signal strength')
    ax.axvline(4.5, color='grey', lw=0.6, ls='--', alpha=0.6)  # midpoint guide

fig.suptitle('Predictive power check — Week 5 Day 4\nMonotonic downward slope = signal has edge', y=1.02)
plt.tight_layout()
plt.savefig('data/week5_day4_predictive_power.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Information Coefficient (IC)

The **IC** is the Spearman rank correlation between today's z-score and the forward residual change.

- IC < 0 means high z predicts negative forward change → convergence → **good**
- IC near 0 → no predictive power
- |IC| > 0.02 with p < 0.01 is typical for a real signal in fixed income

We also compute per-maturity IC to check whether the edge is consistent across the curve.

In [ ]:
print("=== Overall IC ===")
for horizon, col in [(5, 'fwd5_bps'), (20, 'fwd20_bps')]:
    r, p = spearmanr(df['z_score'], df[col])
    sig_flag = '✓' if p < 0.01 else '✗'
    print(f"  {horizon:2d}d  IC = {r:+.4f}   p = {p:.2e}   {sig_flag}")

print("\n=== Per-maturity IC (5-day) ===")
ic_rows = []
for mat, grp in df.groupby('maturity'):
    r5,  p5  = spearmanr(grp['z_score'], grp['fwd5_bps'])
    r20, p20 = spearmanr(grp['z_score'], grp['fwd20_bps'])
    ic_rows.append({'maturity': mat, 'IC_5d': r5, 'p_5d': p5, 'IC_20d': r20, 'p_20d': p20})

ic_df = pd.DataFrame(ic_rows).set_index('maturity')
print(ic_df.round(4).to_string())

## 6. Per-maturity IC bar chart

Visualise whether the signal's predictive power is consistent across maturities, or concentrated in one or two points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, col, label in [
    (axes[0], 'IC_5d',  '5-day IC'),
    (axes[1], 'IC_20d', '20-day IC'),
]:
    colors = ['steelblue' if v < 0 else 'firebrick' for v in ic_df[col]]
    ax.bar(ic_df.index.astype(str), ic_df[col], color=colors, width=0.6)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Maturity (years)')
    ax.set_ylabel('Spearman IC')
    ax.set_title(label)

fig.suptitle('Per-maturity Information Coefficient (negative = good)')
plt.tight_layout()
plt.savefig('data/week5_day4_ic_by_maturity.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conditional mean forward return at extreme z-scores

Focus on just the signal-triggering observations (|z| > 2).  
This is what the backtest will actually trade — so the mean forward return here is the most decision-relevant number.

In [ ]:
cheap = df[df['z_score'] >  2.0]   # signal: buy
rich  = df[df['z_score'] < -2.0]   # signal: sell

print("=== Cheap bonds (z > +2, buy signal) ===")
print(f"  N observations : {len(cheap):,}")
print(f"  Mean fwd 5d    : {cheap['fwd5_bps'].mean():+.3f} bp   (expect negative)")
print(f"  Mean fwd 20d   : {cheap['fwd20_bps'].mean():+.3f} bp   (expect negative)")
print(f"  % fwd5 < 0     : {(cheap['fwd5_bps'] < 0).mean():.1%}")
print(f"  % fwd20 < 0    : {(cheap['fwd20_bps'] < 0).mean():.1%}")

print("\n=== Rich bonds (z < -2, sell signal) ===")
print(f"  N observations : {len(rich):,}")
print(f"  Mean fwd 5d    : {rich['fwd5_bps'].mean():+.3f} bp   (expect positive)")
print(f"  Mean fwd 20d   : {rich['fwd20_bps'].mean():+.3f} bp   (expect positive)")
print(f"  % fwd5 > 0     : {(rich['fwd5_bps'] > 0).mean():.1%}")
print(f"  % fwd20 > 0    : {(rich['fwd20_bps'] > 0).mean():.1%}")

## 8. Verdict

Interpret the results above:

- **Bin plot monotonically slopes down?** → signal has predictive structure  
- **IC significantly negative (p < 0.01)?** → edge is real, not sampling noise  
- **Mean fwd return at |z|>2 has the right sign?** → the tradeable signal points the right way  

If all three pass → proceed to Day 5 (save signal panel).  
If the bin plot is flat → stop, debug the residual computation or demeaning step.